In [22]:
import torch
import torch.nn as nn
import torch.linalg as la
from torch.utils.data import DataLoader
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix, log_loss
import pandas as pd
import os
import requests
import tarfile
from PIL import Image
import sys
from omegaconf import OmegaConf

# Add the project source directory to the Python path
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

from src.models.dnf import DNFNetwork
from src.data.dataset import load_mnist
from src.utils.losses import compute_logits
from src.train import get_target_distributions
from src.utils.evaluation import (
    get_all_predictions,
    get_classification_report_and_cm,
    calculate_ece_and_reliability_diagram,
    calculate_nll_and_brier_score,
    get_ood_confidences_and_plot,
)

# ----------------------------
# Configuration
# ----------------------------
# Manually compose the configuration to mimic Hydra's behavior
conf_path = os.path.join(project_root, 'conf')
cfg = OmegaConf.load(os.path.join(conf_path, 'config.yaml'))

# Load and assign the default configs
cfg.data = OmegaConf.load(os.path.join(conf_path, 'data', 'mnist.yaml'))
cfg.model = OmegaConf.load(os.path.join(conf_path, 'model', 'dnf_cnn.yaml'))
cfg.training = OmegaConf.load(os.path.join(conf_path, 'training', 'default.yaml'))

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# --- IMPORTANT: SET YOUR CHECKPOINT PATH ---
# Update this path to point to your saved model checkpoint file.
CHECKPOINT_PATH = "checkpoint_epoch_50.pth"

# Model Hyperparameters from config (must match the saved model)
NUM_LAYERS = cfg.model.num_layers
HIDDEN_CHANNELS = cfg.model.hidden_channels
NUM_CLASSES = cfg.training.num_classes
FEATURES = cfg.training.features

Using device: cpu


## 1. Setup: Model and Data Definitions
This section contains all the necessary class definitions for the DNF model, data loading functions, and helper functions required for evaluation.

In [ ]:
# ----------------------------
# Data Loading
# ----------------------------
# The load_mnist function from the project returns DataLoaders.
# We need the underlying dataset for some evaluation metrics.
_, test_loader = load_mnist(cfg.data)
test_dataset = test_loader.dataset

print(f"Loaded MNIST test dataset with {len(test_dataset)} samples.")

Loaded MNIST test dataset with 10000 samples.


## 2. Load Model from Checkpoint
Instantiate the model and load the saved weights and latent variable distributions from the specified checkpoint file.

In [39]:
# --- 1. Instantiate a new model ---
model = DNFNetwork(in_channels=1, num_layers=NUM_LAYERS, hidden_channels=HIDDEN_CHANNELS).to(device)

# --- 2. Load the checkpoint file ---
if not os.path.exists(CHECKPOINT_PATH):
    raise FileNotFoundError(f"Checkpoint file not found at '{CHECKPOINT_PATH}'. Please update the path.")

print(f"Loading checkpoint from {CHECKPOINT_PATH}...")
checkpoint = torch.load(CHECKPOINT_PATH, map_location=device)

# --- 3. Load the states into the new instances ---
model.load_state_dict(checkpoint['model_state_dict'])

# Load the trained means and variances for correct evaluation
trainable_means = checkpoint['trainable_means'].to(device)
trainable_log_vars = checkpoint['trainable_log_vars'].to(device)

# --- 4. Set the model to evaluation mode ---
model.eval()

print("Model loaded successfully and set to evaluation mode.")

Loading checkpoint from checkpoint_epoch_50.pth...
Model loaded successfully and set to evaluation mode.


## 3. Evaluation Metrics
Now we compute the model's predictions on the test set and evaluate its performance across different metrics.

In [ ]:
# Get model predictions for the entire test set
print("Computing model predictions on the test set...")
final_target_dists = get_target_distributions(trainable_means, trainable_log_vars, NUM_CLASSES)
y_true, y_pred, probabilities, confidences = get_all_predictions(
    model, test_loader, device, final_target_dists
)
print("Predictions computed.")

Computing model predictions on the test set...


0.01s - Debugger warning: It seems that frozen modules are being used, which may
0.01s - Debugger warning: It seems that frozen modules are being used, which may
0.00s - make the debugger miss breakpoints. Please pass -Xfrozen_modules=off
0.01s - Debugger warning: It seems that frozen modules are being used, which may
0.00s - make the debugger miss breakpoints. Please pass -Xfrozen_modules=off
0.00s - to python to disable frozen modules.
0.00s - Note: Debugging will proceed. Set PYDEVD_DISABLE_FILE_VALIDATION=1 to disable this validation.
0.00s - to python to disable frozen modules.
0.00s - make the debugger miss breakpoints. Please pass -Xfrozen_modules=off
0.00s - Note: Debugging will proceed. Set PYDEVD_DISABLE_FILE_VALIDATION=1 to disable this validation.
0.00s - to python to disable frozen modules.
0.00s - Note: Debugging will proceed. Set PYDEVD_DISABLE_FILE_VALIDATION=1 to disable this validation.
0.01s - Debugger warning: It seems that frozen modules are being used, which may
0

### 3.1. Classification Report & Confusion Matrix

In [ ]:
# Get classification report and confusion matrix
mnist_target_names = [str(i) for i in range(NUM_CLASSES)]
_, _ = get_classification_report_and_cm(y_true, y_pred, mnist_target_names)

### 3.2. Model Calibration (ECE)
Expected Calibration Error measures the difference between a model's confidence and its accuracy. A lower ECE means the model is better calibrated.

In [ ]:
# Calculate ECE and plot reliability diagram
_ = calculate_ece_and_reliability_diagram(confidences, y_pred, y_true)

### 3.3. NLL and Brier Score
These are proper scoring rules that measure the quality of the model's probabilistic predictions.

In [ ]:
# Calculate NLL and Brier score
_, _ = calculate_nll_and_brier_score(y_true, probabilities)

### 3.4. Out-of-Distribution (OOD) Detection
Here, we test the model's confidence on a dataset it has never seen before (notMNIST). A well-behaved model should be less confident on OOD data compared to in-distribution data (MNIST).

In [ ]:
def get_notmnist_loader(batch_size=256):
    """Downloads, extracts, and creates a DataLoader for the notMNIST dataset."""
    root = './data'
    url = 'http://yaroslavvb.com/upload/notMNIST/notMNIST_small.tar.gz'
    filename = 'notMNIST_small.tar.gz'
    filepath = os.path.join(root, filename)
    extract_path = os.path.join(root, 'notMNIST_small')

    if not os.path.exists(extract_path):
        if not os.path.exists(root):
            os.makedirs(root)
        print(f"Downloading {url}...")
        try:
            headers = {'User-Agent': 'Mozilla/5.0'}
            r = requests.get(url, stream=True, timeout=30, headers=headers)
            r.raise_for_status()
            with open(filepath, 'wb') as f:
                for chunk in r.iter_content(chunk_size=8192):
                    f.write(chunk)
            print("Download complete.")
            print(f"Extracting {filepath}...")
            with tarfile.open(filepath, 'r:gz') as tar:
                tar.extractall(path=root)
            print("Extraction complete.")
            os.remove(filepath)
        except Exception as e:
            print(f"Failed to download or extract notMNIST: {e}")
            return None

    transform = transforms.Compose([
        transforms.Grayscale(num_output_channels=1),
        transforms.ToTensor(),
        transforms.Normalize((0.5,), (0.5,))
    ])

    def is_valid_file(path):
        try:
            with Image.open(path) as img:
                img.verify()
            return True
        except Exception:
            return False

    try:
        notmnist_dataset = ImageFolder(root=extract_path, transform=transform, is_valid_file=is_valid_file)
        notmnist_loader = DataLoader(notmnist_dataset, batch_size=batch_size, shuffle=False)
        print(f"notMNIST loaded successfully with {len(notmnist_dataset)} valid images.")
        return notmnist_loader
    except Exception as e:
        print(f"Failed to create notMNIST loader: {e}")
        return None

notmnist_loader = get_notmnist_loader()

In [ ]:
# Get and plot OOD confidences
if notmnist_loader:
    print("\nComputing confidences for out-of-distribution data (notMNIST)...")
    _, _ = get_ood_confidences_and_plot(
        model, test_loader, notmnist_loader, device, final_target_dists
    )
else:
    print("Could not run OOD analysis because notMNIST failed to load.")